Conversion del mapa de Veracruz



In [1]:
# 1. Instalación de librerías necesarias
!pip install rasterio gdal

import rasterio
from rasterio.control import GroundControlPoint
from rasterio.transform import from_gcps
import os

# --- CONFIGURACIÓN DEL USUARIO ---
# Asegúrate de haber subido la imagen del mapa (convertida a .jpg o .png) a tu Drive o local de Colab
input_path = 'mapa_veracruz_extracto.png'
output_path = 'mapa_veracruz_georeferenciado.tif'

# 2. Definición de Puntos de Control (GCPs)
# Debes identificar al menos 3 o 4 cruces (+) en tu mapa.
# GroundControlPoint(row, col, x, y)
# row/col son los píxeles en la imagen (y, x)
# x/y son las coordenadas reales (Longitud, Latitud) que leas en el mapa del INEGI.

gcps = [
    # Ejemplo (debes ajustar estos valores con los que leas en las esquinas del mapa):
    GroundControlPoint(row=500, col=500, x=-96.1333, y=19.1733),
    GroundControlPoint(row=500, col=5000, x=-96.1000, y=19.1733),
    GroundControlPoint(row=4000, col=500, x=-96.1333, y=19.1400),
    GroundControlPoint(row=4000, col=5000, x=-96.1000, y=19.1400)
]

# 3. Cálculo de la transformación
transform = from_gcps(gcps)

# 4. Aplicar la georreferenciación y guardar como GeoTIFF
with rasterio.open(input_path) as src:
    data = src.read()

    # Definimos el CRS (Sistema de Referencia de Coordenadas)
    # Para México e INEGI usualmente es WGS84 (EPSG:4326) o UTM Zona 14N
    crs = 'EPSG:4326'

    with rasterio.open(
        output_path,
        'w',
        driver='GTiff',
        height=data.shape[1],
        width=data.shape[2],
        count=data.shape[0],
        dtype=data.dtype,
        crs=crs,
        transform=transform,
    ) as dst:
        dst.write(data)

print(f"¡Éxito! El mapa georreferenciado se guardó en: {output_path}")

/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)


¡Éxito! El mapa georreferenciado se guardó en: mapa_veracruz_georeferenciado.tif


1. Preparación e Instalación
Primero, instalamos las librerías necesarias para manejar imágenes geoespaciales y procesamiento de visión artificial.

In [2]:
!pip install opencv-python-headless rasterio numpy matplotlib

import cv2
import numpy as np
import rasterio
from matplotlib import pyplot as plt

# Cargar la imagen georreferenciada (archivo .tif generado anteriormente)
path = 'mapa_veracruz_georeferenciado.tif'

with rasterio.open(path) as src:
    # Leer las bandas y convertirlas a un formato que OpenCV entienda (H, W, C)
    img = src.read([1, 2, 3]).transpose(1, 2, 0)
    # Convertir de RGB a BGR para OpenCV
    img_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    # Convertir a HSV para segmentación por color
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    metadata = src.meta

2. Extracción de la Capa Urbana (Zonas Amarillas)


In [3]:
# Definir rango para el color amarillo del mapa INEGI
lower_yellow = np.array([20, 30, 100])
upper_yellow = np.array([35, 255, 255])

# Crear máscara
urban_mask = cv2.inRange(hsv, lower_yellow, upper_yellow)

# Limpieza morfológica (eliminar ruido/puntos pequeños)
kernel = np.ones((5,5), np.uint8)
urban_mask = cv2.morphologyEx(urban_mask, cv2.MORPH_OPEN, kernel)

# Guardar resultado
with rasterio.open('capa_urbana.tif', 'w', **metadata) as dst:
    dst.write(urban_mask, 1)

print("Capa urbana extraída.")

Capa urbana extraída.


3. Extracción de Cuerpos de Agua (Zonas Azules)

In [4]:
# Definir rango para el color azul (agua y retícula)
lower_blue = np.array([90, 50, 50])
upper_blue = np.array([130, 255, 255])

# Crear máscara
water_mask = cv2.inRange(hsv, lower_blue, upper_blue)

# Aplicar un filtro para suavizar los bordes del agua
water_mask = cv2.GaussianBlur(water_mask, (5,5), 0)

# Guardar resultado
with rasterio.open('capa_agua.tif', 'w', **metadata) as dst:
    dst.write(water_mask, 1)

print("Capa de cuerpos de agua extraída.")

Capa de cuerpos de agua extraída.


4. Extracción de Infraestructura Pluvial y Carreteras (Líneas Negras/Grises)

In [5]:
# Definir rango para colores oscuros (negro/gris de carreteras y símbolos)
lower_black = np.array([0, 0, 0])
upper_black = np.array([180, 255, 50])

infra_mask = cv2.inRange(hsv, lower_black, upper_black)

# Guardar resultado
with rasterio.open('capa_infraestructura.tif', 'w', **metadata) as dst:
    dst.write(infra_mask, 1)

print("Capa de infraestructura extraída.")

Capa de infraestructura extraída.
